In [ ]:
# =========================================================
# Clay Classification using Machine Learning
# Models: Decision Tree, Random Forest, SVM
# Validation: LOOCV (good for very small datasets)
# =========================================================

# ===============================
# 1. Import libraries
# ===============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA

np.random.seed(42)

# Optional: SHAP
# If SHAP is not installed, run:
# pip install shap
import shap

# ===============================
# 2. Load dataset
# ===============================
file_path = "/content/Clay classification.csv"   # change path if needed
data = pd.read_csv(file_path)

# Clean column names
data.columns = data.columns.str.strip()

print("Initial shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())
print("\nFirst 5 rows:")
print(data.head())

# ===============================
# 3. Remove missing data
# ===============================
print("\nMissing values before cleaning:")
print(data.isnull().sum())

data = data.dropna()

print("\nShape after dropping missing rows:", data.shape)
print("\nMissing values after cleaning:")
print(data.isnull().sum())

# ===============================
# 4. Define input and output
# ===============================
X = data.drop("Output", axis=1)
y = data["Output"]

print("\nInput features:")
print(X.columns.tolist())

print("\nOutput classes:")
print(y.unique())

# ===============================
# 5. Encode output labels
# ===============================
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("\nClass mapping:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"{cls} -> {i}")

# ===============================
# 6. Basic plots
# ===============================

# 6.1 Class distribution
plt.figure(figsize=(6, 4))
data["Output"].value_counts().plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel("Clay Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 6.2 Correlation heatmap
plt.figure(figsize=(10, 8))
corr = X.corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# 6.3 PCA plot
scaler_for_pca = StandardScaler()
X_scaled_for_pca = scaler_for_pca.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_for_pca)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, s=100)
plt.title("PCA Plot of Clay Samples")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

# ===============================
# 7. Define models
# ===============================
dt_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", DecisionTreeClassifier(random_state=42, max_depth=3))
])

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42))
])

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))
])

models = {
    "Decision Tree": dt_pipeline,
    "Random Forest": rf_pipeline,
    "SVM": svm_pipeline
}

# ===============================
# 8. LOOCV prediction
# ===============================
loo = LeaveOneOut()
results = {}

for model_name, model in models.items():
    y_pred = cross_val_predict(model, X, y_encoded, cv=loo)
    acc = accuracy_score(y_encoded, y_pred)

    results[model_name] = {
        "y_pred": y_pred,
        "accuracy": acc
    }

    print("\n==============================")
    print(f"{model_name} - LOOCV Results")
    print("==============================")
    print("Accuracy:", acc)
    print("\nClassification Report:")
    print(classification_report(y_encoded, y_pred, target_names=label_encoder.classes_))

# ===============================
# 9. Confusion matrices
# ===============================
for model_name in results:
    cm = confusion_matrix(y_encoded, results[model_name]["y_pred"])

    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

# ===============================
# 10. Accuracy comparison plot
# ===============================
model_names = list(results.keys())
accuracies = [results[m]["accuracy"] for m in model_names]

plt.figure(figsize=(7, 5))
plt.bar(model_names, accuracies)
plt.title("Model Accuracy Comparison (LOOCV)")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

# ===============================
# 11. Fit models on full data
# For feature importance / SHAP / final prediction plots
# ===============================
dt_pipeline.fit(X, y_encoded)
rf_pipeline.fit(X, y_encoded)
svm_pipeline.fit(X, y_encoded)

dt_model = dt_pipeline.named_steps["model"]
rf_model = rf_pipeline.named_steps["model"]

# ===============================
# 12. Feature importance
# ===============================
# Decision Tree
dt_importance = pd.Series(dt_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
dt_importance.sort_values().plot(kind="barh")
plt.title("Feature Importance - Decision Tree")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\nDecision Tree Feature Importance:")
print(dt_importance)

# Random Forest
rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
rf_importance.sort_values().plot(kind="barh")
plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\nRandom Forest Feature Importance:")
print(rf_importance)

# ===============================
# 13. SHAP plots
# SHAP works best for tree-based models here
# ===============================
# Scale X once for interpretation consistency with pipeline
scaler_full = StandardScaler()
X_scaled_full = scaler_full.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled_full, columns=X.columns)

# Decision Tree SHAP
explainer_dt = shap.TreeExplainer(dt_model)
shap_values_dt = explainer_dt.shap_values(X_scaled_df)

print("\nGenerating SHAP summary plot for Decision Tree...")
if isinstance(shap_values_dt, list):
    shap.summary_plot(shap_values_dt[0], X_scaled_df, feature_names=X.columns)
else:
    shap.summary_plot(shap_values_dt, X_scaled_df, feature_names=X.columns)

# Random Forest SHAP
explainer_rf = shap.TreeExplainer(rf_model)
shap_values_rf = explainer_rf.shap_values(X_scaled_df)

print("\nGenerating SHAP summary plot for Random Forest...")
if isinstance(shap_values_rf, list):
    shap.summary_plot(shap_values_rf[0], X_scaled_df, feature_names=X.columns)
else:
    shap.summary_plot(shap_values_rf, X_scaled_df, feature_names=X.columns)

# ===============================
# 14. Actual vs predicted plot
# Overall plot to show predicted type
# ===============================
plot_df = pd.DataFrame({
    "Sample": np.arange(1, len(y_encoded) + 1),
    "Actual": y_encoded,
    "Decision Tree": results["Decision Tree"]["y_pred"],
    "Random Forest": results["Random Forest"]["y_pred"],
    "SVM": results["SVM"]["y_pred"]
})

plt.figure(figsize=(10, 6))
plt.plot(plot_df["Sample"], plot_df["Actual"], marker="o", label="Actual", linewidth=2)
plt.plot(plot_df["Sample"], plot_df["Decision Tree"], marker="s", linestyle="--", label="Decision Tree")
plt.plot(plot_df["Sample"], plot_df["Random Forest"], marker="^", linestyle="--", label="Random Forest")
plt.plot(plot_df["Sample"], plot_df["SVM"], marker="d", linestyle="--", label="SVM")

plt.title("Actual vs Predicted Clay Type")
plt.xlabel("Sample Number")
plt.ylabel("Encoded Class")
plt.yticks(
    ticks=np.arange(len(label_encoder.classes_)),
    labels=label_encoder.classes_
)
plt.legend()
plt.tight_layout()
plt.show()

# ===============================
# 15. Robustness check
# Small input noise
# ===============================
X_noisy = X.copy()

for col in X_noisy.columns:
    std_col = X_noisy[col].std()
    noise = np.random.normal(0, 0.05 * std_col, size=len(X_noisy))
    X_noisy[col] = X_noisy[col] + noise

robustness_results = {}

for model_name, model in models.items():
    y_pred_noisy = cross_val_predict(model, X_noisy, y_encoded, cv=loo)
    acc_noisy = accuracy_score(y_encoded, y_pred_noisy)
    robustness_results[model_name] = acc_noisy

print("\n==============================")
print("Robustness Check (Noisy Input)")
print("==============================")
for model_name, acc_noisy in robustness_results.items():
    print(f"{model_name}: {acc_noisy:.4f}")

plt.figure(figsize=(7, 5))
plt.bar(list(robustness_results.keys()), list(robustness_results.values()))
plt.title("Robustness Check Accuracy (Noisy Inputs)")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

# ===============================
# 16. Predict a new sample
# ===============================
new_sample = pd.DataFrame({
    "d90": [20],
    "d10": [2],
    "d50": [8],
    "water absorption": [12],
    "Al/Si": [0.45],
    "SO3": [0.8],
    "CaCO3": [4.5],
    "MgO": [1.2],
    "Na2O": [0.3],
    "Heat": [150]
})

new_sample = new_sample[X.columns]

dt_pred = dt_pipeline.predict(new_sample)
rf_pred = rf_pipeline.predict(new_sample)
svm_pred = svm_pipeline.predict(new_sample)

print("\n==============================")
print("Prediction for New Sample")
print("==============================")
print("Decision Tree prediction:", label_encoder.inverse_transform(dt_pred)[0])
print("Random Forest prediction:", label_encoder.inverse_transform(rf_pred)[0])
print("SVM prediction:", label_encoder.inverse_transform(svm_pred)[0])

